In [1]:
# 1. Ingresamos los datos del caso

import numpy as np
import pandas as pd
 
D = 0.008   # d = tasa de descuento mensual (0.8%)
 
datos = pd.DataFrame({
    'segmento': ['Constantes', 'Casuales'],
    'precio':   [20, 12],      # P = precio de hoy
    'costo':    [8, 8],        # C = costo por cliente
    'unidades': [1000, 2000],  # Q = unidades al mes hoy
    'eps_x':    [0.5, 0.2],    # εx = efecto de SU precio en las unidades del OTRO segmento
    'churn':    [0.03, 0.06],  # c = churn mensual
}).set_index('segmento')
datos

,precio,costo,unidades,eps_x,churn
segmento,,,,,
Constantes,20,8,1000,0.5,0.03
Casuales,12,8,2000,0.2,0.06


In [2]:
# 2. Pruebas de precio anteriores: a cada precio, cuántas unidades se vendieron al mes
pruebas = pd.DataFrame({
    'segmento': ['Constantes'] * 5 + ['Casuales'] * 5,
    'precio':   [18, 19, 20, 21, 22,  10.8, 11.4, 12, 12.6, 13.2],
    'unidades': [1159, 1074, 1000, 934, 875,  2603, 2274, 2000, 1770, 1576],
})
pruebas

,segmento,precio,unidades
0,Constantes,18.0,1159
1,Constantes,19.0,1074
2,Constantes,20.0,1000
3,Constantes,21.0,934
4,Constantes,22.0,875
5,Casuales,10.8,2603
6,Casuales,11.4,2274
7,Casuales,12.0,2000
8,Casuales,12.6,1770
9,Casuales,13.2,1576


In [3]:
# 3. EJEMPLO resuelto: elasticidad de Constantes
g = pruebas[pruebas.segmento == 'Constantes']
 
# Pendiente de la recta ln(unidades) contra ln(precio): es la elasticidad ε (como en el Demo 1)
eps_const = np.polyfit(np.log(g.precio), np.log(g.unidades), 1)[0]
print(f'Elasticidad de Constantes: {eps_const:.2f}')

Elasticidad de Constantes: -1.40


In [5]:
# 4. TU TURNO: elasticidad de Casuales
g = pruebas[pruebas.segmento == 'Casuales']
 
# TU CÓDIGO AQUÍ  (calculen eps_casu con la misma fórmula y escríbanla con print)
eps_casu = np.polyfit(np.log(g.precio), np.log(g.unidades), 1)[0]
print(f'Elasticidad de Casuales: {eps_casu:.2f}')

datos['eps'] = [round(eps_const, 1), round(eps_casu, 1)]   # ε de cada segmento, a un decimal
datos

Elasticidad de Casuales: -2.50


,precio,costo,unidades,eps_x,churn,eps
segmento,,,,,,
Constantes,20,8,1000,0.5,0.03,-1.4
Casuales,12,8,2000,0.2,0.06,-2.5


In [6]:
# Precio óptimo y restricciones
# 5. Las mismas funciones del Demo 2 de Ritmo (no hay que cambiarlas)
def precio_optimo(C, eps):
    return C * eps / (1 + eps)                 # P* = C × ε ÷ (1 + ε)
 
def aplicar_reglas(p_hoy, p_opt, C):
    p = min(p_opt, p_hoy * 1.10)               # tope: sube máximo 10%
    p = max(p, p_hoy * 0.90)                   # tope: baja máximo 10%
    p = max(p, 1.2 * C)                        # margen mínimo: precio ≥ 1.2 × C
    return p
 
def utilidad(p, p_hoy, C, Q, eps):
    unidades = Q * (p / p_hoy) ** eps          # cuántas unidades se venden a p
    return (p - C) * unidades                  # utilidad = (p − C) × unidades
 
def clv(m, c, d=D):
    return m * (1 - c) / (c + d)               # CLV = m × (1 − c) ÷ (c + d)
 
def churn_maximo(m_nuevo, clv_hoy, d=D):
    return (m_nuevo - clv_hoy * d) / (m_nuevo + clv_hoy)   # c'

In [12]:
# 6. Precio de Constantes. EJEMPLO resuelto: segmento Constantes
s = datos.loc['Constantes']
 
p_opt  = precio_optimo(s.costo, s.eps)
p_prop = aplicar_reglas(s.precio, p_opt, s.costo)
 
print(f'Precio óptimo P*:          ${p_opt:.2f}')
print(f'Precio propuesto (reglas): ${p_prop:.2f}')
print(f'Utilidad hoy:      {utilidad(s.precio, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad propuesta:{utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad óptima:{utilidad(p_opt, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Costo restricción:  {utilidad(p_opt, s.precio, s.costo, s.unidades, s.eps) - utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')

Precio óptimo P*:          $28.00
Precio propuesto (reglas): $22.00
Utilidad hoy:      12,000
Utilidad propuesta:12,251
Utilidad óptima:12,487
Costo restricción:  236


In [13]:
# 7. Precio de Casuales. EJEMPLO resuelto: segmento Casuales
s = datos.loc['Casuales']
 
p_opt  = precio_optimo(s.costo, s.eps)
p_prop = aplicar_reglas(s.precio, p_opt, s.costo)
 
print(f'Precio óptimo P*:          ${p_opt:.2f}')
print(f'Precio propuesto (reglas): ${p_prop:.2f}')
print(f'Utilidad hoy:      {utilidad(s.precio, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad propuesta:{utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad óptima:{utilidad(p_opt, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Costo restricción:  {utilidad(p_opt, s.precio, s.costo, s.unidades, s.eps) - utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')


Precio óptimo P*:          $13.33
Precio propuesto (reglas): $13.20
Utilidad hoy:      8,000
Utilidad propuesta:8,195
Utilidad óptima:8,197
Costo restricción:  2


In [14]:
# 8. La función nueva: utilidad de los DOS segmentos juntos, con elasticidad cruzada
def utilidad_total(p_const, p_casu):
    a, b = datos.loc['Constantes'], datos.loc['Casuales']
    q_const = a.unidades * (p_const / a.precio) ** a.eps * (p_casu / b.precio) ** b.eps_x
    q_casu  = b.unidades * (p_casu / b.precio) ** b.eps * (p_const / a.precio) ** a.eps_x
    u_const = (p_const - a.costo) * q_const
    u_casu  = (p_casu - b.costo) * q_casu
    return u_const, u_casu
 
# EJEMPLO resuelto: sube SOLO Constantes a $22 (Casuales se queda en $12)
hoy = utilidad_total(20, 12)
sube = utilidad_total(22, 12)
print(f'Constantes: {hoy[0]:,.0f} -> {sube[0]:,.0f}')
print(f'Casuales:   {hoy[1]:,.0f} -> {sube[1]:,.0f}')
print(f'Total:      {sum(hoy):,.0f} -> {sum(sube):,.0f}')

Constantes: 12,000 -> 12,251
Casuales:   8,000 -> 8,390
Total:      20,000 -> 20,642


In [15]:
# 9. TU TURNO: suben LOS DOS segmentos (Constantes a $22 y Casuales a $13.20)
# TU CÓDIGO AQUÍ  (calculen utilidad_total(22, 13.2) e impriman la utilidad de cada segmento y el total)
hoy = utilidad_total(20, 12)
ambos = utilidad_total(22, 13.2)

print(f'Constantes: {hoy[0]:,.0f} -> {ambos[0]:,.0f}')
print(f'Casuales:   {hoy[1]:,.0f} -> {ambos[1]:,.0f}')
print(f'Total:      {sum(hoy):,.0f} -> {sum(ambos):,.0f}')

# TU CÓDIGO AQUÍ  (comparen con la suma de las dos utilidades propuestas de las celdas 3 y 4: 12,251 + 8,195 = 20,446)
total_separado = 12251 + 8195
total_con_cruce = sum(ambos)

print(f'Total separando segmentos: {total_separado:,.0f}')
print(f'Total con elasticidad cruzada: {total_con_cruce:,.0f}')
print(f'Diferencia: {total_con_cruce - total_separado:,.0f}')

Constantes: 12,000 -> 12,487
Casuales:   8,000 -> 8,595
Total:      20,000 -> 21,082
Total separando segmentos: 20,446
Total con elasticidad cruzada: 21,082
Diferencia: 636


In [16]:
# 10. Churn máximo
# Háganlo para los dos segmentos

# Constantes
s = datos.loc['Constantes']
# 1) clv_hoy = clv(m, c)           con m = precio de hoy − costo
clv_const = clv(s.precio - s.costo, s.churn)
# 2) m_nuevo = p_prop − costo
m_const_nuevo = 22 - s.costo
# 3) churn_maximo(m_nuevo, clv_hoy)
cmax_const = churn_maximo(m_const_nuevo, clv_const)

print(f'Constantes - CLV hoy: {clv_const:.2f}')
print(f'Constantes - Margen nuevo: {m_const_nuevo:.2f}')
print(f'Constantes - Churn máximo: {cmax_const:.2%}')

# Casuales
s = datos.loc['Casuales']
# 1) clv_hoy = clv(m, c)           con m = precio de hoy − costo
clv_casu = clv(s.precio - s.costo, s.churn)
# 2) m_nuevo = p_prop − costo
m_casu_nuevo = 13.2 - s.costo
# 3) churn_maximo(m_nuevo, clv_hoy)
cmax_casu = churn_maximo(m_casu_nuevo, clv_casu)

print(f'Casuales - CLV hoy: {clv_casu:.2f}')
print(f'Casuales - Margen nuevo: {m_casu_nuevo:.2f}')
print(f'Casuales - Churn máximo: {cmax_casu:.2%}')

Constantes - CLV hoy: 306.32
Constantes - Margen nuevo: 14.00
Constantes - Churn máximo: 3.61%
Casuales - CLV hoy: 55.29
Casuales - Margen nuevo: 5.20
Casuales - Churn máximo: 7.86%
